In [0]:
table_name = "dim_facilities"
silver_table_fqn = f"hdfc_ergo.hdfc_silver.{table_name}"

df = spark.read.table("hdfc_ergo.hdfc_bronze.dim_facilities")
display(df.limit(5))

In [0]:
from pyspark.sql.functions import col, upper, trim, lower, regexp_replace, when, to_date, row_number, abs as spark_abs
from pyspark.sql.window import Window

# 1. Standardize Text: UPPER(TRIM())
text_cols = ["facility_id", "hospital_license_number", "facility_name", "facility_type", 
             "address", "city", "state", "pincode", "accreditation", 
             "trauma_center_level", "emr_system", "network_status", 
             "patient_safety_grade", "active_status"]
for c in text_cols:
    df = df.withColumn(c, upper(trim(col(c))))

df = df.withColumn("email", lower(trim(col("email"))))

# 2. Clean Phone (Remove non-numeric)
df = df.withColumn("phone", regexp_replace(col("phone"), "[^0-9]", ""))

# 3. Boolean Mapping
for bool_col in ["teaching_hospital", "emergency_department", "icu_available"]:
    df = df.withColumn(bool_col, 
                       when(upper(trim(col(bool_col))).isin("YES", "1", "TRUE"), True)
                       .when(upper(trim(col(bool_col))).isin("NO", "0", "FALSE"), False)
                       .otherwise(None).cast("boolean"))

# 4. Safe Cast Integers
for int_col in ["beds_count", "surgical_suites"]:
    df = df.withColumn(int_col, trim(col(int_col)).cast("int"))

# 5. Precision Optimization (Safe Decimal Cast to avoid text crashes)
for dec_col in ["cost_index", "nabh_rating", "readmission_rate", "mortality_rate"]:
    is_valid_decimal = trim(col(dec_col)).rlike("^[+-]?([0-9]+\\.?[0-9]*|[0-9]*\\.?[0-9]+)$")
    df = df.withColumn(dec_col, when(is_valid_decimal, trim(col(dec_col)).cast("decimal(15,2)")).otherwise(None))
    if dec_col in ["cost_index"]:
        df = df.withColumn(dec_col, spark_abs(col(dec_col))) # ABS for financial/cost metrics

# 6. Safe Cast Dates
for date_col in ["record_created_date", "record_modified_date"]:
    df = df.withColumn(date_col, to_date(trim(col(date_col))))

# 7. Deduplication (PARTITION BY facility_id)
# NOTE: Using _inserted_at instead of _ingested_at based on your actual data
dedup_window = Window.partitionBy("facility_id").orderBy(col("_inserted_at").desc())
df = df.withColumn("_dq_is_deduped", row_number().over(dedup_window))
df = df.filter(col("_dq_is_deduped") == 1).drop("_dq_is_deduped")

# 8. Drop audit columns and source surrogate keys
df = df.drop("_inserted_at", "_source_file", "facility_sk")

display(df.limit(5))

In [0]:
df.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(silver_table_fqn)

print(f"✅ Successfully wrote {table_name} to Silver layer.")